# Cálculo y Cálculo Vectorial para Machine Learning

**Objetivo**: Dominar los conceptos fundamentales del cálculo escalar y vectorial con visualizaciones Python, desde funciones escalares hasta los teoremas de Green, Stokes y Gauss.

**Estructura del notebook**:
1. Funciones escalares y vectoriales
2. Curvas y superficies paramétricas
3. Operador nabla: gradiente, divergencia, curl, Laplaciano
4. Integrales de línea, dobles, triples y de superficie
5. Teoremas de Green, Stokes y Gauss
6. Aplicaciones en ML (descenso de gradiente, PINNs, etc.)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)


In [ ]:
# Instalación e importaciones
# !pip install numpy matplotlib scipy sympy -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy import integrate
import sympy as sp
from sympy import symbols, sin, cos, exp, ln, diff, Matrix, latex
import warnings
warnings.filterwarnings('ignore')
sp.init_printing(use_unicode=True)

plt.rcParams.update({'font.size': 11, 'figure.dpi': 100, 'figure.figsize': (12, 5)})
print("Librerías cargadas correctamente.")
print(f"NumPy {np.__version__} | SymPy {sp.__version__}")


## 1. Funciones Escalares: $f: \mathbb{R}^n \to \mathbb{R}$

Una función escalar asigna un número real a cada punto del espacio.

### Ejemplos fundamentales

| Función | Fórmula | Relevancia ML |
|---------|---------|---------------|
| Paraboloide | $f(x,y) = x^2 + y^2$ | Función de pérdida convexa |
| Rosenbrock | $f(x,y) = (1-x)^2 + 100(y-x^2)^2$ | Benchmark de optimización |
| Gaussiana | $f(x,y) = e^{-(x^2+y^2)/2}$ | Kernel RBF, distribución normal |
| Seno 2D | $f(x,y) = \sin(x)\cos(y)$ | Activaciones periódicas |

### Conceptos clave
- **Curvas de nivel**: $\{(x,y) : f(x,y) = c\}$ — análogo a curvas de pérdida constante
- **Punto crítico**: $\nabla f = \mathbf{0}$ — mínimo, máximo o punto de silla
- **Hessiano**: $H_{ij} = \partial^2 f / \partial x_i \partial x_j$ — curvatura local


In [ ]:
# ── Visualización de funciones escalares ─────────────────────────────────
x_range = np.linspace(-3, 3, 200)
y_range = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x_range, y_range)

funciones = {
    'Paraboloide\n$x^2 + y^2$': X**2 + Y**2,
    'Rosenbrock\n$(1-x)^2+100(y-x^2)^2$': (1-X)**2 + 100*(Y - X**2)**2,
    'Gaussiana\n$e^{-(x^2+y^2)/2}$': np.exp(-(X**2 + Y**2)/2),
    'Seno 2D\n$\\sin(x)\\cos(y)$': np.sin(X)*np.cos(Y),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for i, (nombre, Z) in enumerate(funciones.items()):
    # Superficie 3D
    ax3d = fig.add_subplot(2, 4, i+1, projection='3d')
    surf = ax3d.plot_surface(X, Y, Z, cmap='viridis', alpha=0.85, linewidth=0)
    ax3d.set_title(nombre, fontsize=9, fontweight='bold')
    ax3d.set_xlabel('x', fontsize=8); ax3d.set_ylabel('y', fontsize=8)
    ax3d.tick_params(labelsize=7)

    # Curvas de nivel
    ax2d = axes[1, i]
    n_levels = 15
    cp = ax2d.contourf(X, Y, Z, levels=n_levels, cmap='viridis', alpha=0.8)
    ax2d.contour(X, Y, Z, levels=n_levels, colors='white', alpha=0.4, linewidths=0.5)
    plt.colorbar(cp, ax=ax2d, shrink=0.8)
    ax2d.set_title(f'Curvas de nivel', fontsize=9)
    ax2d.set_xlabel('x', fontsize=8); ax2d.set_ylabel('y', fontsize=8)

plt.suptitle('Funciones Escalares $f: \mathbb{R}^2 \to \mathbb{R}$',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 2. Campos Vectoriales: $\mathbf{F}: \mathbb{R}^n \to \mathbb{R}^n$

Un campo vectorial asigna un vector a cada punto del espacio.

### Ejemplos

| Campo | Fórmula | Interpretación |
|-------|---------|----------------|
| Radial (fuente) | $\mathbf{F} = (x, y)$ | Fluido emanando del origen |
| Rotacional | $\mathbf{F} = (-y, x)$ | Vórtice, campo magnético |
| Gradiente | $\mathbf{F} = \nabla f$ | Dirección de máxima subida |
| Uniforme | $\mathbf{F} = (1, 0)$ | Viento, campo eléctrico uniforme |

Un campo se llama **conservativo** si $\mathbf{F} = \nabla f$ para alguna $f$ (potencial).


In [ ]:
# ── Campos vectoriales 2D ─────────────────────────────────────────────────
X_q, Y_q = np.meshgrid(np.linspace(-2, 2, 15), np.linspace(-2, 2, 15))
r = np.sqrt(X_q**2 + Y_q**2) + 1e-10

campos = {
    'Radial $\\mathbf{F}=(x,y)$\n(fuente, div>0)': (X_q, Y_q),
    'Rotacional $\\mathbf{F}=(-y,x)$\n(vórtice, div=0)': (-Y_q, X_q),
    'Dipolo\n$\\mathbf{F}=(x,-y)$': (X_q, -Y_q),
    'Punto silla\n$\\mathbf{F}=(y,x)$': (Y_q, X_q),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (nombre, (U, V)) in zip(axes, campos.items()):
    mag = np.sqrt(U**2 + V**2) + 1e-10
    im = ax.quiver(X_q, Y_q, U/mag, V/mag, mag, cmap='plasma',
                   scale=20, width=0.004, pivot='mid')
    plt.colorbar(im, ax=ax, shrink=0.7, label='|F|')
    ax.set_title(nombre, fontsize=10, fontweight='bold')
    ax.set_xlim(-2.3, 2.3); ax.set_ylim(-2.3, 2.3)
    ax.set_aspect('equal')
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle('Campos Vectoriales $\mathbf{F}: \mathbb{R}^2 \to \mathbb{R}^2$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Curvas Paramétricas: $\mathbf{r}: \mathbb{R} \to \mathbb{R}^n$

Una curva paramétrica describe una trayectoria: $\mathbf{r}(t) = (x(t), y(t), z(t))$.

### Conceptos asociados
- **Vector tangente**: $\mathbf{r}'(t) = (x'(t), y'(t), z'(t))$
- **Longitud de arco**: $s = \int_a^b \|\mathbf{r}'(t)\|\,dt$
- **Curvatura**: $\kappa = \|\mathbf{r}''\| / \|\mathbf{r}'\|^2$ (para curvas planas)
- **Vector unitario tangente**: $\hat{\mathbf{T}} = \mathbf{r}'/\|\mathbf{r}'\|$


In [ ]:
# ── Curvas paramétricas ───────────────────────────────────────────────────
t = np.linspace(0, 2*np.pi, 1000)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Círculo
ax = axes[0]
xc, yc = np.cos(t), np.sin(t)
ax.plot(xc, yc, 'b-', lw=2)
# Tangentes en varios puntos
for ti in [0, np.pi/3, 2*np.pi/3, np.pi, 4*np.pi/3, 5*np.pi/3]:
    x0, y0 = np.cos(ti), np.sin(ti)
    tx, ty = -np.sin(ti), np.cos(ti)
    ax.annotate('', xy=(x0+0.3*tx, y0+0.3*ty), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.set_title('Círculo\nr(t)=(cos t, sin t)', fontweight='bold')

# Cicloide
ax = axes[1]
t2 = np.linspace(0, 4*np.pi, 1000)
xcy = t2 - np.sin(t2)
ycy = 1 - np.cos(t2)
ax.plot(xcy, ycy, 'g-', lw=2)
ax.fill_between(xcy, ycy, alpha=0.2, color='green')
ax.set_title('Cicloide\nr(t)=(t-sin t, 1-cos t)', fontweight='bold')
ax.grid(alpha=0.3); ax.set_aspect('equal')

# Hélice 3D
ax = fig.add_subplot(1, 4, 3, projection='3d')
t3 = np.linspace(0, 4*np.pi, 500)
xh = np.cos(t3); yh = np.sin(t3); zh = t3/(4*np.pi)
ax.plot(xh, yh, zh, 'purple', lw=2)
# Tangentes en hélice
for ti in np.linspace(0, 4*np.pi, 8):
    i = int(ti / (4*np.pi) * 499)
    if i < 490:
        tx = -np.sin(t3[i]); ty = np.cos(t3[i]); tz = 1/(4*np.pi)
        norm = np.sqrt(tx**2+ty**2+tz**2)
        ax.quiver(xh[i], yh[i], zh[i], tx/norm, ty/norm, tz/norm,
                  length=0.15, color='red', arrow_length_ratio=0.4)
ax.set_title('Hélice 3D', fontsize=10, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

# Longitud de arco
ax = axes[3]
curves = {
    'Círculo (r=1)': (np.cos(t), np.sin(t), 2*np.pi),
    'Elipse (a=2,b=1)': (2*np.cos(t), np.sin(t), None),
    'Cardioide': ((1+np.cos(t))*np.cos(t), (1+np.cos(t))*np.sin(t), None),
}
for nombre, (xc2, yc2, L_teo) in curves.items():
    dx = np.diff(xc2); dy = np.diff(yc2)
    L_num = np.sum(np.sqrt(dx**2 + dy**2))
    label = f'{nombre}\nL≈{L_num:.3f}'
    if L_teo:
        label += f' (teo: {L_teo:.3f})'
    ax.plot(xc2, yc2, lw=2, label=label)
ax.legend(fontsize=8); ax.set_aspect('equal')
ax.set_title(r'Longitud de Arco' + '\n' + r'$s=\int\|r\prime\|dt$', fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Curvas Paramétricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Superficies Paramétricas: $\mathbf{r}: \mathbb{R}^2 \to \mathbb{R}^3$

Una superficie paramétrica mapea un dominio 2D al espacio 3D: $\mathbf{r}(u,v) = (x(u,v), y(u,v), z(u,v))$.

### Vector normal y elemento de área
$$\mathbf{r}_u = \frac{\partial\mathbf{r}}{\partial u}, \quad \mathbf{r}_v = \frac{\partial\mathbf{r}}{\partial v}$$
$$d\mathbf{S} = (\mathbf{r}_u \times \mathbf{r}_v)\,du\,dv, \quad dS = \|\mathbf{r}_u \times \mathbf{r}_v\|\,du\,dv$$


In [ ]:
# ── Superficies paramétricas ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 5))

# Esfera
ax1 = fig.add_subplot(141, projection='3d')
u_s = np.linspace(0, 2*np.pi, 50)
v_s = np.linspace(0, np.pi, 30)
U, V = np.meshgrid(u_s, v_s)
Xs = np.sin(V)*np.cos(U); Ys = np.sin(V)*np.sin(U); Zs = np.cos(V)
ax1.plot_surface(Xs, Ys, Zs, cmap='Blues', alpha=0.8)
ax1.set_title('Esfera\nr=(sinv cosu, sinv sinu, cosv)', fontsize=9, fontweight='bold')

# Toro
ax2 = fig.add_subplot(142, projection='3d')
R_t, r_t = 2, 0.7
Xt = (R_t + r_t*np.cos(V))*np.cos(U)
Yt = (R_t + r_t*np.cos(V))*np.sin(U)
Zt = r_t*np.sin(V)
ax2.plot_surface(Xt, Yt, Zt, cmap='Oranges', alpha=0.8)
ax2.set_title('Toro\n(R=2, r=0.7)', fontsize=9, fontweight='bold')

# Paraboloide
ax3 = fig.add_subplot(143, projection='3d')
u_p = np.linspace(-2, 2, 40); v_p = np.linspace(-2, 2, 40)
Up, Vp = np.meshgrid(u_p, v_p)
Zp = Up**2 + Vp**2
ax3.plot_surface(Up, Vp, Zp, cmap='Greens', alpha=0.8)
ax3.set_title('Paraboloide\nz=x²+y²', fontsize=9, fontweight='bold')

# Silla de montar con normales
ax4 = fig.add_subplot(144, projection='3d')
u_sd = np.linspace(-1.5, 1.5, 30); v_sd = np.linspace(-1.5, 1.5, 30)
Usd, Vsd = np.meshgrid(u_sd, v_sd)
Zsd = Usd**2 - Vsd**2
ax4.plot_surface(Usd, Vsd, Zsd, cmap='RdBu', alpha=0.6)
# Normales
for ui, vi in [(-1,0),(0,-1),(1,0),(0,1),(0,0)]:
    zi = ui**2 - vi**2
    # Normal = (-2u, 2v, 1) / norm
    n = np.array([-2*ui, 2*vi, 1])
    n = n / np.linalg.norm(n)
    ax4.quiver(ui, vi, zi, n[0], n[1], n[2], length=0.5, color='red', arrow_length_ratio=0.4)
ax4.set_title('Silla\nz=x²-y² + normales', fontsize=9, fontweight='bold')

plt.suptitle('Superficies Paramétricas $\mathbf{r}(u,v): \mathbb{R}^2 \to \mathbb{R}^3$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 6. El Operador Nabla $\nabla$

### 6.1 Definición
El operador nabla (también llamado **del**) es un operador diferencial vectorial formal:

$$\nabla = \hat{\mathbf{i}}\frac{\partial}{\partial x} + \hat{\mathbf{j}}\frac{\partial}{\partial y} + \hat{\mathbf{k}}\frac{\partial}{\partial z}
= \begin{pmatrix} \partial/\partial x \\ \partial/\partial y \\ \partial/\partial z \end{pmatrix}$$

Es un vector de operadores diferenciales. Según **cómo lo combinamos**, obtenemos:

| Operación | Expresión | Resultado | Nombre |
|-----------|-----------|-----------|--------|
| $\nabla f$ | $\nabla$ aplicado a escalar $f$ | Campo vectorial | **Gradiente** |
| $\nabla \cdot \mathbf{F}$ | Producto punto con campo $\mathbf{F}$ | Función escalar | **Divergencia** |
| $\nabla \times \mathbf{F}$ | Producto cruz con campo $\mathbf{F}$ | Campo vectorial | **Rotacional** |
| $\nabla \cdot \nabla f = \nabla^2 f$ | Divergencia del gradiente | Función escalar | **Laplaciano** |

### 6.2 Mnemónico visual

```
       Función escalar f
            │
       ─────┼───────── ∇f  →  GRADIENTE  (vector)
            │
       Campo vectorial F
         ───┤
       ∇·F ─┤──── DIVERGENCIA  (escalar)
       ∇×F ─┘──── ROTACIONAL   (vector)
            │
       ∇·∇f ──── LAPLACIANO    (escalar)
```

---
## 7. Gradiente — $\nabla f$

### 7.1 Definición
El **gradiente** de una función escalar $f: \mathbb{R}^n \to \mathbb{R}$ es el campo vectorial:

$$\nabla f = \left(\frac{\partial f}{\partial x},\; \frac{\partial f}{\partial y},\; \frac{\partial f}{\partial z}\right)$$

### 7.2 Interpretaciones fundamentales
1. **Dirección de máximo crecimiento**: $\nabla f(\mathbf{p})$ apunta en la dirección en que $f$ crece más rápido desde $\mathbf{p}$
2. **Magnitud de cambio**: $|\nabla f(\mathbf{p})|$ es la tasa de cambio máxima
3. **Perpendicular a las curvas de nivel**: $\nabla f \perp$ curvas $f = c$
4. **Derivada direccional**: $D_{\hat{u}} f = \nabla f \cdot \hat{u}$

### 7.3 Propiedad geométrica fundamental
> Las **curvas de nivel** (isosuperficies) de $f$ son siempre **perpendiculares** al campo $\nabla f$.  
> Esto es central en: optimización, ray-tracing, física del potencial, etc.

### 7.4 Reglas de cálculo (análogas al cálculo 1D)

$$\nabla(fg) = f\nabla g + g\nabla f \qquad \nabla(f/g) = \frac{g\nabla f - f\nabla g}{g^2}$$
$$\nabla(f \circ g) = f'(g)\,\nabla g \quad (\text{regla de la cadena})$$

### 7.5 Ejemplo analítico
Sea $f(x,y) = x^2 + 4y^2$. Entonces:
$$\nabla f = (2x,\, 8y)$$
En $(1, 1)$: $\nabla f = (2, 8)$, magnitud $= \sqrt{4+64} \approx 8.25$.  
La función crece más rápido en la dirección $(2,8)$.

In [ ]:
# ─── Gradiente: visualización completa ────────────────────────────────────

x = np.linspace(-3, 3, 300)
y = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(x, y)
Z = X**2 + 4*Y**2  # Elipsoide

# Gradiente numérico sobre grilla gruesa
xg = np.linspace(-3, 3, 14)
yg = np.linspace(-3, 3, 14)
Xg, Yg = np.meshgrid(xg, yg)
Ug = 2*Xg          # ∂f/∂x = 2x
Vg = 8*Yg          # ∂f/∂y = 8y
mag = np.sqrt(Ug**2 + Vg**2) + 1e-10
Ug_n, Vg_n = Ug/mag, Vg/mag  # Normalizado para visualizar dirección

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Superficie + gradiente (proyección)
ax1 = axes[0]
cs = ax1.contourf(X, Y, Z, 20, cmap='plasma')
plt.colorbar(cs, ax=ax1)
ax1.contour(X, Y, Z, 10, colors='white', linewidths=0.6, alpha=0.5)
ax1.quiver(Xg, Yg, Ug_n, Vg_n, mag,
           cmap='Wistia', scale=20, width=0.005, alpha=0.9)
ax1.set_title('$f = x^2 + 4y^2$: curvas de nivel + $\nabla f$ (flechas)', fontsize=11)
ax1.set_aspect('equal'); ax1.set_xlabel('x'); ax1.set_ylabel('y')

# Anotar que gradiente ⊥ contornos
ax1.text(-2.8, 2.7, '$\nabla f \perp$ curvas de nivel', color='yellow',
         fontsize=10, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))

# 2. Solo gradiente coloreado por magnitud
ax2 = axes[1]
q = ax2.quiver(Xg, Yg, Ug, Vg, mag, cmap='hot', scale=150, width=0.005)
plt.colorbar(q, ax=ax2, label='$|\nabla f|$')
ax2.set_title('Campo gradiente $\nabla f = (2x, 8y)$', fontsize=11)
ax2.set_aspect('equal'); ax2.set_xlabel('x'); ax2.set_ylabel('y')
ax2.axhline(0, color='k', lw=0.5); ax2.axvline(0, color='k', lw=0.5)

# 3. Perfil 1D: dirección de máximo crecimiento desde (1,1)
ax3 = axes[2]
p0 = np.array([1.0, 1.0])
grad_p0 = np.array([2*p0[0], 8*p0[1]])
theta_max = np.arctan2(grad_p0[1], grad_p0[0])

# Función en diferentes direcciones desde p0
thetas = np.linspace(0, 2*np.pi, 300)
derivadas = 2*p0[0]*np.cos(thetas) + 8*p0[1]*np.sin(thetas)

ax3.plot(np.degrees(thetas), derivadas, 'b-', lw=2)
ax3.axhline(0, color='k', lw=0.8, linestyle='--')
ax3.axvline(np.degrees(theta_max), color='red', lw=2, linestyle='--',
            label=f'$\theta_{{max}}$={np.degrees(theta_max):.1f}°')
ax3.axvline(np.degrees(theta_max)+180, color='red', lw=2, linestyle='--')
ax3.fill_between(np.degrees(thetas), derivadas, 0,
                 where=(derivadas > 0), alpha=0.3, color='green', label='Crece')
ax3.fill_between(np.degrees(thetas), derivadas, 0,
                 where=(derivadas < 0), alpha=0.3, color='red', label='Decrece')
ax3.set_xlabel('Dirección $\theta$ (°)'); ax3.set_ylabel('$D_{\hat{u}} f$')
ax3.set_title('Derivada direccional desde $(1,1)$ en función del ángulo', fontsize=10)
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)
max_val = np.linalg.norm(grad_p0)
ax3.text(30, max_val*0.8, f'Máximo: $|\nabla f|$={max_val:.2f}', color='green')

plt.suptitle('Gradiente $\nabla f$: dirección de máximo crecimiento, perpendicular a contornos',
             fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

# Verificación simbólica con SymPy
x_s, y_s = sp.symbols('x y')
f_sym = x_s**2 + 4*y_s**2
grad_sym = [sp.diff(f_sym, var) for var in [x_s, y_s]]
print(f"SymPy: ∇f = {grad_sym}")
print(f"Dirección máx en (1,1): {[g.subs([(x_s,1),(y_s,1)]) for g in grad_sym]}")

---
## 8. Divergencia — $\nabla \cdot \mathbf{F}$

### 8.1 Definición
La **divergencia** de un campo vectorial $\mathbf{F} = (F_1, F_2, F_3)$ es la función escalar:

$$\nabla \cdot \mathbf{F} = \frac{\partial F_1}{\partial x} + \frac{\partial F_2}{\partial y} + \frac{\partial F_3}{\partial z}$$

### 8.2 Interpretación física
> La divergencia mide la **tendencia del campo a expandirse o contraerse** en un punto.

- $\nabla \cdot \mathbf{F} > 0$: el punto es una **fuente** (el fluido sale de ahí)
- $\nabla \cdot \mathbf{F} < 0$: el punto es un **sumidero** (el fluido entra)
- $\nabla \cdot \mathbf{F} = 0$: campo **solenoidal** (sin fuentes ni sumideros)

### 8.3 Campos solenoïdales
Un campo con divergencia nula en todo $\mathbb{R}^3$ describe conservación del flujo.  
Ejemplo: campo magnético $\nabla \cdot \mathbf{B} = 0$ (Ley de Gauss para magnetismo).

### 8.4 Ejemplos analíticos

| Campo $\mathbf{F}$ | $\nabla \cdot \mathbf{F}$ | Tipo |
|----|----|----|
| $(x, y, z)$ | $3$ | Fuente uniforme |
| $(-x, -y, -z)$ | $-3$ | Sumidero uniforme |
| $(-y, x, 0)$ | $0$ | Solenoidal (remolino) |
| $(x^2, y^2, 0)$ | $2x + 2y$ | Fuentes variables |

In [ ]:
# ─── Divergencia: fuentes, sumideros y campos solenoïdales ───────────────

x = np.linspace(-3, 3, 14)
y = np.linspace(-3, 3, 14)
X, Y = np.meshgrid(x, y)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

casos = [
    ('Campo radial (fuente)\n$\mathbf{F}=(x,y)$, $\nabla\cdot\mathbf{F}=2$',
     X, Y, 2*np.ones_like(X)),
    ('Campo convergente (sumidero)\n$\mathbf{F}=(-x,-y)$, $\nabla\cdot\mathbf{F}=-2$',
     -X, -Y, -2*np.ones_like(X)),
    ('Remolino (solenoidal)\n$\mathbf{F}=(-y,x)$, $\nabla\cdot\mathbf{F}=0$',
     -Y, X, np.zeros_like(X)),
    ('Cuadrático\n$\mathbf{F}=(x^2,y^2)$, $\nabla\cdot\mathbf{F}=2x+2y$',
     X**2, Y**2, 2*X + 2*Y),
    ('Dipolo\n$\mathbf{F}=(x,-y)$, $\nabla\cdot\mathbf{F}=0$',
     X, -Y, np.zeros_like(X)),
    ('Campo mixto\n$\mathbf{F}=(x^2-y^2, 2xy)$',
     X**2 - Y**2, 2*X*Y, 2*X + 2*Y),
]

for ax, (titulo, U, V, div) in zip(axes.flat, casos):
    # Mapa de calor de la divergencia
    x_fine = np.linspace(-3, 3, 200)
    y_fine = np.linspace(-3, 3, 200)
    Xf, Yf = np.meshgrid(x_fine, y_fine)
    # Recalcular div fina según el caso
    if 'radial' in titulo: div_fine = 2*np.ones_like(Xf)
    elif 'convergente' in titulo: div_fine = -2*np.ones_like(Xf)
    elif 'Remolino' in titulo or 'Dipolo' in titulo: div_fine = np.zeros_like(Xf)
    elif 'Cuadr' in titulo: div_fine = 2*Xf + 2*Yf
    elif 'mixto' in titulo: div_fine = 2*Xf + 2*Yf
    else: div_fine = np.zeros_like(Xf)

    cf = ax.contourf(Xf, Yf, div_fine, 20, cmap='RdBu_r', alpha=0.7)
    plt.colorbar(cf, ax=ax, label='$\nabla\cdot\mathbf{F}$')
    mag = np.sqrt(U**2 + V**2) + 1e-10
    ax.quiver(X, Y, U/mag, V/mag, color='black', scale=20, width=0.004, alpha=0.8)
    ax.set_title(titulo, fontsize=9)
    ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle('Divergencia $\nabla \cdot \mathbf{F}$: rojo=fuente (+), azul=sumidero (−)',
             fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

---
## 9. Rotacional (Curl) — $\nabla \times \mathbf{F}$

### 9.1 Definición
El **rotacional** de un campo $\mathbf{F} = (F_1, F_2, F_3)$ es el campo vectorial:

$$\nabla \times \mathbf{F} = \begin{vmatrix} \hat{\mathbf{i}} & \hat{\mathbf{j}} & \hat{\mathbf{k}} \\ \partial_x & \partial_y & \partial_z \\ F_1 & F_2 & F_3 \end{vmatrix}
= \left(\frac{\partial F_3}{\partial y} - \frac{\partial F_2}{\partial z},\;
  \frac{\partial F_1}{\partial z} - \frac{\partial F_3}{\partial x},\;
  \frac{\partial F_2}{\partial x} - \frac{\partial F_1}{\partial y}\right)$$

### 9.2 En 2D (componente $z$ del rotacional)
Para $\mathbf{F} = (P(x,y), Q(x,y), 0)$:

$$\text{curl}_z = \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}$$

### 9.3 Interpretación física
> El rotacional mide la **tendencia del campo a girar** alrededor de un punto.  
> Si imaginas una pequeña rueda de paletas en el fluido: el rotacional indica si girarÍa y en qué sentido.

- $\nabla \times \mathbf{F} \neq 0$: hay **rotación local** (campo no conservativo)
- $\nabla \times \mathbf{F} = 0$: campo **irrotacional** (conservativo si es simplemente conexo)

### 9.4 Campos conservativos
$\mathbf{F}$ es conservativo si existe $f$ tal que $\mathbf{F} = \nabla f$.  
Condición necesaria: $\nabla \times \mathbf{F} = 0$.

### 9.5 Ejemplos

| Campo | $\text{curl}_z$ | Tipo |
|-------|---------|------|
| $(-y, x)$ | $2$ | Rotación pura |
| $(x, y)$ | $0$ | Irrotacional |
| $(-y, x)/(x^2+y^2)$ | $0$ (salvo origen) | Pseudo-rotacional |
| $(x^2, xy)$ | $y$ | Rotacional variable |

In [ ]:
# ─── Rotacional: visualización en 2D y 3D ─────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

x = np.linspace(-3, 3, 13)
y = np.linspace(-3, 3, 13)
X, Y = np.meshgrid(x, y)

x_f = np.linspace(-3, 3, 200); y_f = np.linspace(-3, 3, 200)
Xf, Yf = np.meshgrid(x_f, y_f)

casos_curl = [
    ('Rotación pura\n$(-y,x)$, $\text{curl}_z=2$',
     -Y, X, 2*np.ones_like(Xf)),
    ('Irrotacional (fuente)\n$(x,y)$, $\text{curl}_z=0$',
     X, Y, np.zeros_like(Xf)),
    ('Flujo cizallado\n$(y,0)$, $\text{curl}_z=-1$',
     Y, np.zeros_like(X), -np.ones_like(Xf)),
    ('Rotacional variable\n$(x^2, xy)$, $\text{curl}_z=y$',
     X**2, X*Y, Yf),
    ('Vórtice libre\n$(-y,x)/(x^2+y^2+0.1)$, $\approx 0$',
     -Y/(X**2+Y**2+0.1), X/(X**2+Y**2+0.1),
     np.zeros_like(Xf)),
    ('Campo dipolo\n$(xy, x^2/2)$, $\text{curl}_z=0$',
     X*Y, X**2/2, np.zeros_like(Xf)),
]

for ax, (titulo, U, V, curl_fine) in zip(axes.flat, casos_curl):
    # Fondo: mapa de calor del rotacional
    lim = max(abs(curl_fine.min()), abs(curl_fine.max()), 0.1)
    cf = ax.contourf(Xf, Yf, curl_fine, 20, cmap='bwr',
                     vmin=-lim, vmax=lim, alpha=0.7)
    plt.colorbar(cf, ax=ax, label='$\nabla\times\mathbf{F}$')
    # Flechas del campo
    mag = np.sqrt(U**2 + V**2) + 1e-10
    ax.quiver(X, Y, U/mag, V/mag, color='black', scale=20, width=0.004, alpha=0.85)
    ax.set_title(titulo, fontsize=9)
    ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.axhline(0, color='k', lw=0.4); ax.axvline(0, color='k', lw=0.4)

plt.suptitle('Rotacional $\nabla\times\mathbf{F}$: rojo=giro antihor. (+), azul=horario (−)',
             fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# Verificación simbólica
x_s, y_s, z_s = sp.symbols('x y z')
F1, F2, F3 = -y_s, x_s, sp.Integer(0)
curl_z = sp.diff(F2, x_s) - sp.diff(F1, y_s)
print(f"SymPy: curl_z de (-y, x, 0) = {curl_z}")
F1b, F2b = x_s**2, x_s*y_s
curl_z_b = sp.diff(F2b, x_s) - sp.diff(F1b, y_s)
print(f"SymPy: curl_z de (x², xy) = {curl_z_b}")

---
## 10. Laplaciano — $\Delta f = \nabla^2 f = \nabla \cdot \nabla f$

### 10.1 Definición
El **Laplaciano** es la divergencia del gradiente:

$$\Delta f = \nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2} + \frac{\partial^2 f}{\partial z^2}$$

### 10.2 Interpretación: comparación con vecinos
> En el punto $\mathbf{p}$, $\Delta f(\mathbf{p})$ mide cuánto difiere $f(\mathbf{p})$ del **promedio de los vecinos**.

- $\Delta f > 0$: el valor en $\mathbf{p}$ es **menor** que la media de sus vecinos → tendencia a subir (mínimo local)
- $\Delta f < 0$: el valor en $\mathbf{p}$ es **mayor** que la media → tendencia a bajar (máximo local)
- $\Delta f = 0$: **función armónica** → el valor en cada punto es exactamente la media de sus vecinos

### 10.3 Ecuaciones con el Laplaciano

| Ecuación | Forma | Física |
|----------|-------|--------|
| **Laplace** | $\Delta f = 0$ | Temperatura estacionaria, electrostática |
| **Poisson** | $\Delta f = g$ | Con fuentes de calor/carga |
| **Difusión** | $\partial_t f = k\Delta f$ | Conducción de calor |
| **Onda** | $\partial_{tt} f = c^2 \Delta f$ | Propagación de ondas |

### 10.4 Aproximación discreta
En una grilla de paso $h$:

$$\Delta f(x_i, y_j) \approx \frac{f_{i+1,j} + f_{i-1,j} + f_{i,j+1} + f_{i,j-1} - 4f_{i,j}}{h^2}$$

Es el conocido **stencil de 5 puntos** usado en diferencias finitas.

In [ ]:
# ─── Laplaciano: visualización y aproximación discreta ─────────────────────

x = np.linspace(-3, 3, 200); y = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x, y)

funciones_lap = {
    'Gaussiana $e^{-(x^2+y^2)}$':
        (np.exp(-(X**2+Y**2)),
         np.exp(-(X**2+Y**2)) * (4*(X**2+Y**2) - 4)),
    'Paraboloide $x^2+y^2$ ($\Delta f=4$)':
        (X**2 + Y**2, 4*np.ones_like(X)),
    'Armónica $x^2-y^2$ ($\Delta f=0$)':
        (X**2 - Y**2, np.zeros_like(X)),
    '$\sin(x)\sin(y)$ ($\Delta f=-2\sin x\sin y$)':
        (np.sin(X)*np.sin(Y), -2*np.sin(X)*np.sin(Y)),
}

fig, axes = plt.subplots(len(funciones_lap), 3,
                         figsize=(18, 5*len(funciones_lap)))

for row, (titulo, (f, lap_f)) in enumerate(funciones_lap.items()):
    # 1. Campo escalar
    ax = axes[row, 0]
    cf = ax.contourf(X, Y, f, 20, cmap='plasma')
    plt.colorbar(cf, ax=ax)
    ax.set_title(f'$f$: {titulo}', fontsize=9)
    ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')

    # 2. Laplaciano
    ax2 = axes[row, 1]
    lim = np.percentile(np.abs(lap_f), 98) + 1e-10
    cf2 = ax2.contourf(X, Y, lap_f, 20, cmap='RdBu_r', vmin=-lim, vmax=lim)
    plt.colorbar(cf2, ax=ax2)
    ax2.set_title(f'$\Delta f$', fontsize=10)
    ax2.set_aspect('equal'); ax2.set_xlabel('x'); ax2.set_ylabel('y')

    # 3. Perfil 1D (y=0)
    ax3 = axes[row, 2]
    mid = len(x)//2
    ax3.plot(x, f[mid, :], 'b-', lw=2, label='$f(x,0)$')
    ax3.plot(x, lap_f[mid, :], 'r-', lw=2, label='$\Delta f(x,0)$')
    ax3.axhline(0, color='k', lw=0.7, linestyle='--')
    ax3.set_xlabel('x'); ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)
    ax3.set_title('Perfil en $y=0$', fontsize=10)

plt.suptitle('Laplaciano $\Delta f = \partial^2_x f + \partial^2_y f$',
             fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# ─── Aproximación discreta del Laplaciano ──────────────────────────────────
print("\n=== Laplaciano discreto (stencil 5 puntos) ===")
N = 50; h = 6/(N-1)
x_d = np.linspace(-3, 3, N); y_d = np.linspace(-3, 3, N)
Xd, Yd = np.meshgrid(x_d, y_d)
f_d = np.exp(-(Xd**2 + Yd**2))

# Stencil: (f[i+1,j] + f[i-1,j] + f[i,j+1] + f[i,j-1] - 4*f[i,j]) / h^2
lap_disc = np.zeros_like(f_d)
lap_disc[1:-1, 1:-1] = (f_d[2:, 1:-1] + f_d[:-2, 1:-1] +
                         f_d[1:-1, 2:] + f_d[1:-1, :-2] -
                         4*f_d[1:-1, 1:-1]) / h**2
lap_analitico = np.exp(-(Xd**2+Yd**2)) * (4*(Xd**2+Yd**2) - 4)
err = np.abs(lap_disc - lap_analitico)[2:-2, 2:-2]
print(f"Error máximo del stencil discreto: {err.max():.4e}")

fig, axes2 = plt.subplots(1, 3, figsize=(16, 4))
for ax, data, title in zip(axes2,
    [f_d, lap_analitico, lap_disc],
    ['$f = e^{-(x^2+y^2)}$', '$\Delta f$ analítico', '$\Delta f$ discreto']):
    im = ax.imshow(data, origin='lower', extent=[-3,3,-3,3], cmap='RdBu_r',
                   vmin=np.percentile(lap_analitico, 2),
                   vmax=np.percentile(lap_analitico, 98))
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=10)
plt.suptitle('Laplaciano: analítico vs discreto', fontsize=12)
plt.tight_layout(); plt.show()

---
## 11. Tabla Comparativa Conceptual

### 11.1 ¿Qué tipo de objeto matemático tenemos?

| Concepto | Dominio | Codominio | Objeto geométrico | Ejemplo matemático | Ejemplo físico |
|----------|---------|-----------|-------------------|--------------------|----------------|
| **Función escalar** | $\mathbb{R}^n$ | $\mathbb{R}$ | Superficie/hipersuperficie | $f(x,y)=x^2+y^2$ | Temperatura |
| **Campo vectorial** | $\mathbb{R}^n$ | $\mathbb{R}^m$ | Flecha en cada punto | $\mathbf{F}(x,y)=(-y,x)$ | Velocidad del fluido |
| **Curva parametrizada** | $\mathbb{R}$ | $\mathbb{R}^n$ | Trayectoria 1D | $\mathbf{r}(t)=(\cos t, \sin t)$ | Posición de partícula |
| **Superficie parametriz.** | $\mathbb{R}^2$ | $\mathbb{R}^3$ | Variedad 2D | $\mathbf{r}(u,v)=(\cos u\sin v,...)$ | Membrana, cáscara |

### 11.2 ¿Qué hacen los operadores diferenciales?

| Operador | Entrada | Salida | Mide |
|----------|---------|--------|------|
| $\nabla f$ (Gradiente) | Escalar $f$ | Vector | Dirección/magnitud de máx. crecimiento |
| $\nabla \cdot \mathbf{F}$ (Divergencia) | Vector $\mathbf{F}$ | Escalar | Expansión/contracción en un punto |
| $\nabla \times \mathbf{F}$ (Rotacional) | Vector $\mathbf{F}$ | Vector | Tendencia al giro |
| $\Delta f$ (Laplaciano) | Escalar $f$ | Escalar | Diferencia con la media vecinal |

### 11.3 Identidades vectoriales importantes

$$\nabla \times (\nabla f) = \mathbf{0} \qquad \text{(el rotacional de un gradiente siempre es cero)}$$
$$\nabla \cdot (\nabla \times \mathbf{F}) = 0 \qquad \text{(la divergencia de un rotacional siempre es cero)}$$
$$\Delta f = \nabla \cdot \nabla f \qquad \text{(laplaciano = divergencia del gradiente)}$$
$$\nabla \times (\nabla \times \mathbf{F}) = \nabla(\nabla \cdot \mathbf{F}) - \Delta \mathbf{F}$$

In [ ]:
# ─── Verificación numérica de identidades vectoriales ─────────────────────

print("=== Verificación: curl(grad f) = 0 ===")
import numpy as np

h = 0.01
x0, y0, z0 = 1.2, 0.7, 0.5

def f(x,y,z): return np.sin(x)*np.exp(y) + z**2

# Gradiente numérico de f
grad_x = (f(x0+h,y0,z0) - f(x0-h,y0,z0)) / (2*h)
grad_y = (f(x0,y0+h,z0) - f(x0,y0-h,z0)) / (2*h)
grad_z = (f(x0,y0,z0+h) - f(x0,y0,z0-h)) / (2*h)

# Definir G = grad f como campo vectorial
G1 = lambda x,y,z: np.cos(x)*np.exp(y)
G2 = lambda x,y,z: np.sin(x)*np.exp(y)
G3 = lambda x,y,z: 2*z

# curl(G) = curl(grad f) ── debe ser cero
# curl_x = dG3/dy - dG2/dz
curl_x = (G3(x0,y0+h,z0)-G3(x0,y0-h,z0))/(2*h) - (G2(x0,y0,z0+h)-G2(x0,y0,z0-h))/(2*h)
curl_y = (G1(x0,y0,z0+h)-G1(x0,y0,z0-h))/(2*h) - (G3(x0+h,y0,z0)-G3(x0-h,y0,z0))/(2*h)
curl_z = (G2(x0+h,y0,z0)-G2(x0-h,y0,z0))/(2*h) - (G1(x0,y0+h,z0)-G1(x0,y0-h,z0))/(2*h)
print(f"  curl(grad f) = ({curl_x:.2e}, {curl_y:.2e}, {curl_z:.2e}) ≈ (0,0,0) ✓")

print("\n=== Verificación: div(curl F) = 0 ===")
F1 = lambda x,y,z: np.sin(x*y)
F2 = lambda x,y,z: np.cos(y*z)
F3 = lambda x,y,z: np.exp(x*z)

# curl F
c1 = (F3(x0,y0+h,z0)-F3(x0,y0-h,z0))/(2*h) - (F2(x0,y0,z0+h)-F2(x0,y0,z0-h))/(2*h)
c2 = (F1(x0,y0,z0+h)-F1(x0,y0,z0-h))/(2*h) - (F3(x0+h,y0,z0)-F3(x0-h,y0,z0))/(2*h)
c3 = (F2(x0+h,y0,z0)-F2(x0-h,y0,z0))/(2*h) - (F1(x0,y0+h,z0)-F1(x0,y0-h,z0))/(2*h)

# div(curl F)
C1 = lambda x,y,z: ((F3(x,y+h,z)-F3(x,y-h,z))/(2*h) - (F2(x,y,z+h)-F2(x,y,z-h))/(2*h))
C2 = lambda x,y,z: ((F1(x,y,z+h)-F1(x,y,z-h))/(2*h) - (F3(x+h,y,z)-F3(x-h,y,z))/(2*h))
C3 = lambda x,y,z: ((F2(x+h,y,z)-F2(x-h,y,z))/(2*h) - (F1(x,y+h,z)-F1(x,y-h,z))/(2*h))
div_curl = ((C1(x0+h,y0,z0)-C1(x0-h,y0,z0))/(2*h) +
            (C2(x0,y0+h,z0)-C2(x0,y0-h,z0))/(2*h) +
            (C3(x0,y0,z0+h)-C3(x0,y0,z0-h))/(2*h))
print(f"  div(curl F) = {div_curl:.2e} ≈ 0 ✓")
print("\nEstas identidades son exactas en cualquier campo diferenciable.")

---
## 12. Integrales — Visión General

### La idea unificadora
> **Integrar = acumular una cantidad sobre un dominio**

En cálculo multivariable el **dominio** puede ser muy variado:

| Dominio | Dimensión | Tipo de integral | Notación |
|---------|-----------|-----------------|---------|
| Intervalo $[a,b]$ | 1D | **Simple** | $\int_a^b f\,dx$ |
| Curva $C$ | 1D (en $\mathbb{R}^n$) | **De línea** | $\int_C f\,ds$ o $\int_C \mathbf{F}\cdot d\mathbf{r}$ |
| Región plana $D$ | 2D | **Doble** | $\iint_D f\,dA$ |
| Región 3D $V$ | 3D | **Triple** | $\iiint_V f\,dV$ |
| Superficie $S$ | 2D (en $\mathbb{R}^3$) | **De superficie** | $\iint_S f\,dS$ o $\iint_S \mathbf{F}\cdot d\mathbf{S}$ |

### Elementos diferenciales
- $ds = |\mathbf{r}'(t)|\,dt$ — elemento de arco de curva
- $dA = dx\,dy$ — elemento de área plana
- $dV = dx\,dy\,dz$ — elemento de volumen
- $dS = |\mathbf{r}_u \times \mathbf{r}_v|\,du\,dv$ — elemento de área superficial
- $d\mathbf{S} = (\mathbf{r}_u \times \mathbf{r}_v)\,du\,dv$ — elemento vectorial (con normal)

In [ ]:
# ─── Diagrama visual de tipos de integrales ───────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# --- 1. Integral simple ---
ax = axes[0, 0]
x = np.linspace(0, 3, 300)
f = np.sin(x) + 1.5
ax.fill_between(x, 0, f, alpha=0.35, color='steelblue')
ax.plot(x, f, 'b-', lw=2)
ax.axhline(0, color='k', lw=0.8)
ax.set_title('Integral simple $\int_0^3 f(x)\,dx$\nÁrea bajo la curva', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
val = np.trapz(f, x)
ax.text(1.5, 0.3, f'$\approx$ {val:.3f}', ha='center', fontsize=12, color='blue',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.grid(True, alpha=0.3)

# --- 2. Integral de línea ---
ax = axes[0, 1]
t = np.linspace(0, 2*np.pi, 300)
xc, yc = np.cos(t), np.sin(t)
f_curva = 0.8 + 0.5*np.sin(3*t)
# Color = valor de f en la curva
sc = ax.scatter(xc, yc, c=f_curva, cmap='hot', s=10, zorder=3)
plt.colorbar(sc, ax=ax, label='f(r(t))')
ax.plot(xc, yc, 'k-', lw=0.5, alpha=0.5)
ax.set_aspect('equal'); ax.set_title('Integral de línea $\int_C f\,ds$\nAcumular $f$ a lo largo de $C$', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.grid(True, alpha=0.3)

# --- 3. Integral doble ---
ax = axes[0, 2]
x_d = np.linspace(-2, 2, 100); y_d = np.linspace(-2, 2, 100)
Xd, Yd = np.meshgrid(x_d, y_d)
Zd = np.exp(-(Xd**2 + Yd**2))
mask = Xd**2 + Yd**2 <= 4
Zd_masked = np.where(mask, Zd, np.nan)
cf = ax.contourf(Xd, Yd, Zd_masked, 15, cmap='plasma')
plt.colorbar(cf, ax=ax)
circle = plt.Circle((0,0), 2, fill=False, color='white', lw=2)
ax.add_patch(circle)
ax.set_aspect('equal'); ax.set_title('Integral doble $\iint_D f\,dA$\nAcumular $f$ sobre región 2D', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y')

# --- 4. Integral de línea vectorial (trabajo) ---
ax = axes[1, 0]
t_w = np.linspace(0, 2*np.pi, 300)
xw, yw = np.cos(t_w), np.sin(t_w)
Fw_x = -yw; Fw_y = xw
# F·dr = (-y dx/dt + x dy/dt) dt
dot = Fw_x * (-np.sin(t_w)) + Fw_y * np.cos(t_w)
ax.plot(xw, yw, 'b-', lw=2, zorder=3)
# Colorear por F·dr
sc2 = ax.scatter(xw, yw, c=dot, cmap='RdYlGn', s=15, zorder=4)
plt.colorbar(sc2, ax=ax, label='$\mathbf{F}\cdot d\mathbf{r}$')
# Algunas flechas de F
tp_sub = np.linspace(0, 2*np.pi, 12, endpoint=False)
for tp in tp_sub:
    px, py = np.cos(tp), np.sin(tp)
    ax.annotate('', xy=(px-0.3*np.sin(tp), py+0.3*np.cos(tp)), xytext=(px, py),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax.set_aspect('equal'); ax.set_title('Integral de línea vectorial $\int_C \mathbf{F}\cdot d\mathbf{r}$\nTrabajo/circulación', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.grid(True, alpha=0.3)

# --- 5. Integral triple (volumen) ---
ax = axes[1, 1]
from mpl_toolkits.mplot3d import Axes3D
ax.remove()
ax = fig.add_subplot(2, 3, 5, projection='3d')
u = np.linspace(0, 2*np.pi, 30); v = np.linspace(0, np.pi, 20)
U, V = np.meshgrid(u, v); R = 1.5
Xs = R*np.sin(V)*np.cos(U); Ys = R*np.sin(V)*np.sin(U); Zs = R*np.cos(V)
ax.plot_surface(Xs, Ys, Zs, alpha=0.3, color='cyan', edgecolor='k', linewidth=0.2)
ax.set_title('Integral triple $\iiint_V f\,dV$\nAcumular en volumen 3D', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

# --- 6. Integral de superficie (flujo) ---
ax = axes[1, 2]
ax.remove()
ax = fig.add_subplot(2, 3, 6, projection='3d')
u = np.linspace(0, 2*np.pi, 25); v = np.linspace(-1.5, 1.5, 20)
U, V = np.meshgrid(u, v)
Xc = np.cos(U); Yc = np.sin(U); Zc = V
# Campo F = (x, 0, 0) → flujo = cos²(u) en el cilindro
ax.plot_surface(Xc, Yc, Zc, alpha=0.3, color='orange', edgecolor='k', linewidth=0.1)
# Normales
u_n = np.linspace(0, 2*np.pi, 8, endpoint=False); v_n = np.array([-1, 0, 1])
Un, Vn = np.meshgrid(u_n, v_n)
ax.quiver(np.cos(Un), np.sin(Un), Vn,
          np.cos(Un)*0.4, np.sin(Un)*0.4, np.zeros_like(Un),
          color='red', arrow_length_ratio=0.3)
ax.set_title('Integral de superficie $\iint_S \mathbf{F}\cdot d\mathbf{S}$\nFlujo a través de superficie', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

plt.suptitle('Los cinco tipos de integrales en cálculo vectorial', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

---
## 13. Integral Simple $\int_a^b f(x)\,dx$

### Definición (Riemann)
$$\int_a^b f(x)\,dx = \lim_{n\to\infty} \sum_{i=1}^n f(x_i^*) \Delta x$$

Acumula el valor de $f$ a lo largo del intervalo $[a,b]$.

### Interpretación
- Si $f \geq 0$: área bajo la curva
- En general: acumulación con signo
- **Teorema Fundamental**: $\int_a^b f'(x)\,dx = f(b) - f(a)$

---
## 14. Integral de Línea de Función Escalar $\int_C f\,ds$

### Definición
Dada $\mathbf{r}(t): [a,b] \to \mathbb{R}^n$ parametrizando $C$:

$$\int_C f\,ds = \int_a^b f(\mathbf{r}(t))\,|\mathbf{r}'(t)|\,dt$$

Aquí $ds = |\mathbf{r}'(t)|\,dt$ es el **elemento de longitud de arco**.

### Interpretación física
- $f = $ densidad lineal → integral = **masa del alambre**
- $f = 1$ → integral = **longitud de la curva**

### Propiedad
**No depende de la orientación** de la curva (el alambre tiene la misma masa recorriéndolo de cualquier sentido).

---
## 15. Integral de Línea de Campo Vectorial $\int_C \mathbf{F}\cdot d\mathbf{r}$

### Definición
$$\int_C \mathbf{F}\cdot d\mathbf{r} = \int_a^b \mathbf{F}(\mathbf{r}(t))\cdot \mathbf{r}'(t)\,dt$$

Aquí $d\mathbf{r} = \mathbf{r}'(t)\,dt$ es el **vector diferencial de desplazamiento**.

### Interpretación física
- $\mathbf{F} = $ fuerza → integral = **trabajo** realizado por $\mathbf{F}$ a lo largo de $C$
- $\mathbf{F} = $ campo de velocidad → integral = **circulación**

### Diferencia clave con $\int_C f\,ds$
| | $\int_C f\,ds$ | $\int_C \mathbf{F}\cdot d\mathbf{r}$ |
|-|---|---|
| Integrand | Escalar $f$ | $\mathbf{F}\cdot d\mathbf{r}$ (producto punto) |
| Orientación | Independiente | Cambia de signo al invertir la curva |
| Unidades | [f]·[longitud] | [F]·[longitud] |

In [ ]:
# ─── Integrales de línea: ejemplos concretos ──────────────────────────────

import numpy as np

# === INTEGRAL ESCALAR: masa de un alambre circular ===
print("=== Integral de línea escalar: masa del alambre ===")
# Curva: circunferencia x²+y²=1, densidad f(x,y)=1+x²
# r(t)=(cos t, sin t), |r'|=1, t∈[0,2π]
t = np.linspace(0, 2*np.pi, 10000)
x_c, y_c = np.cos(t), np.sin(t)
f_densidad = 1 + x_c**2          # Densidad variable
ds_dt = np.ones_like(t)           # |r'(t)| = 1 para circunferencia de radio 1
masa = np.trapz(f_densidad * ds_dt, t)
print(f"  Masa = ∫_C (1+x²)ds = {masa:.4f}")
print(f"  Exacto: ∫_0^2π (1+cos²t)dt = 2π + π = 3π = {3*np.pi:.4f}")

# Longitud de la curva (f=1)
long = np.trapz(ds_dt, t)
print(f"  Longitud = ∫_C ds = {long:.4f} (exacto: 2π = {2*np.pi:.4f})")

# === INTEGRAL VECTORIAL: trabajo ===
print("\n=== Integral de línea vectorial: trabajo ===")
# Campo F(x,y) = (-y, x)  sobre la circunferencia
F1 = -y_c; F2 = x_c
dx_dt = -np.sin(t); dy_dt = np.cos(t)     # r'(t)
integrando = F1*dx_dt + F2*dy_dt
trabajo = np.trapz(integrando, t)
print(f"  Trabajo = ∫_C F·dr = {trabajo:.4f}")
print(f"  Exacto: ∫_0^2π (sin²t + cos²t)dt = 2π = {2*np.pi:.4f}")

# Campo conservativo: F = (x, y) = ∇(x²/2 + y²/2)
F1_c = x_c; F2_c = y_c
integrando_c = F1_c*dx_dt + F2_c*dy_dt
trabajo_c = np.trapz(integrando_c, t)
print(f"  Campo conservativo (∮ sobre curva cerrada) = {trabajo_c:.6f} ≈ 0")
print("  → Campo conservativo: trabajo nulo en curva cerrada")

# === Visualización ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Integral escalar: masa
ax = axes[0]
sc = ax.scatter(x_c, y_c, c=f_densidad, cmap='hot', s=8)
plt.colorbar(sc, ax=ax, label='$f(x,y) = 1+x^2$')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title(f'Integral escalar\n$\int_C(1+x^2)ds \approx {masa:.3f}$', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y')

# 2. Integral vectorial: F·dr (trabajo)
ax = axes[1]
sc2 = ax.scatter(x_c, y_c, c=integrando, cmap='RdYlGn', s=8)
plt.colorbar(sc2, ax=ax, label=r'$\mathbf{F}\cdot\mathbf{r}^{\prime}$')
t_arr = np.linspace(0, 2*np.pi, 12, endpoint=False)
for tp in t_arr:
    px, py = np.cos(tp), np.sin(tp)
    dx, dy = -np.sin(tp), np.cos(tp)
    fvx, fvy = -py, px
    ax.annotate('', xy=(px+0.25*fvx, py+0.25*fvy), xytext=(px, py),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title(f'Campo $\mathbf{{F}}=(-y,x)$ — trabajo={trabajo:.3f}', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y')

# 3. Trayectoria curvilínea: trabajo a lo largo de hélice
ax3 = fig.add_subplot(133, projection='3d')
t3 = np.linspace(0, 2*np.pi, 500)
xh, yh, zh = np.cos(t3), np.sin(t3), 0.5*t3/(2*np.pi)
# Campo F=(0,0,1) (gravedad inversa)
Fhx, Fhy, Fhz = np.zeros_like(t3), np.zeros_like(t3), np.ones_like(t3)
dxh, dyh, dzh = -np.sin(t3), np.cos(t3), 0.5/(2*np.pi)*np.ones_like(t3)
integ_helix = Fhx*dxh + Fhy*dyh + Fhz*dzh
trab_helix = np.trapz(integ_helix, t3)
sc3 = ax3.scatter(xh, yh, zh, c=integ_helix, cmap='plasma', s=5)
ax3.set_title(f'Hélice con $\mathbf{{F}}=(0,0,1)$\ntrabajo={trab_helix:.4f}', fontsize=10)
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('z')

plt.suptitle('Integrales de Línea: escalar y vectorial', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
## 16. Integral Doble $\iint_D f(x,y)\,dA$

### Definición (suma de Riemann 2D)
$$\iint_D f(x,y)\,dA = \lim_{\Delta x,\Delta y \to 0} \sum_{i,j} f(x_i, y_j)\,\Delta x\,\Delta y$$

### Interpretación
- $f = $ densidad superficial → integral = **masa de la lámina**
- $f = $ altura → integral = **volumen bajo la superficie**
- $f = 1$ → integral = **área de la región**

### Cálculo por Fubini (orden de integración)
$$\iint_D f\,dA = \int_a^b \int_{g_1(x)}^{g_2(x)} f(x,y)\,dy\,dx$$

---
## 17. Integral Triple $\iiint_V f(x,y,z)\,dV$

$$\iiint_V f(x,y,z)\,dV = \int \int \int f\,dx\,dy\,dz$$

- $f = \rho$ (densidad) → integral = **masa del sólido**
- $f = 1$ → integral = **volumen de $V$**

---
## 18. Integral de Superficie Escalar $\iint_S f\,dS$

Dada la parametrización $\mathbf{r}(u,v)$:

$$\iint_S f\,dS = \iint_D f(\mathbf{r}(u,v))\,|\mathbf{r}_u \times \mathbf{r}_v|\,du\,dv$$

donde $|\mathbf{r}_u \times \mathbf{r}_v|$ es el **factor de área superficial**.

---
## 19. Integral de Flujo $\iint_S \mathbf{F}\cdot d\mathbf{S}$

$$\iint_S \mathbf{F}\cdot d\mathbf{S} = \iint_D \mathbf{F}(\mathbf{r}(u,v))\cdot(\mathbf{r}_u \times \mathbf{r}_v)\,du\,dv$$

Mide cuánto **flujo** de $\mathbf{F}$ atraviesa $S$ en dirección normal.

- $\mathbf{F} = $ velocidad del fluido → flujo volumétrico
- $\mathbf{F} = $ campo eléctrico → flujo eléctrico

In [ ]:
# ─── Integrales doble, triple y de superficie ─────────────────────────────

import numpy as np
from scipy import integrate as sci_int

# === 1. INTEGRAL DOBLE ===
print("=== Integral doble: ∬_D xy dA, D = [0,2]x[0,1] ===")
f_doble = lambda y, x: x*y
resultado, err = sci_int.dblquad(f_doble, 0, 2, 0, 1)
print(f"  scipy: {resultado:.4f} (exacto: 1.0)")

# Disco unitario
print("\n∬_{x²+y²≤1} (x²+y²) dA — usando polar:")
def f_polar(r, theta): return r**2 * r  # f=r², dA=r dr dθ
res_polar, _ = sci_int.dblquad(f_polar, 0, 2*np.pi, 0, 1)
print(f"  Resultado: {res_polar:.4f} (exacto: π/2 = {np.pi/2:.4f})")

# === 2. INTEGRAL TRIPLE ===
print("\n=== Integral triple: ∭_V dV sobre la esfera de radio R ===")
R_sph = 2.0
def f_vol(z, y, x): return 1.0
from scipy.integrate import tplquad
res_vol, _ = tplquad(f_vol,
                      -R_sph, R_sph,
                      lambda x: -np.sqrt(max(R_sph**2-x**2, 0)),
                      lambda x: np.sqrt(max(R_sph**2-x**2, 0)),
                      lambda x,y: -np.sqrt(max(R_sph**2-x**2-y**2, 0)),
                      lambda x,y: np.sqrt(max(R_sph**2-x**2-y**2, 0)))
print(f"  Volumen esfera R=2: {res_vol:.4f} (exacto: 4πR³/3 = {4*np.pi*R_sph**3/3:.4f})")

# === 3. INTEGRAL DE SUPERFICIE: área de la esfera ===
print("\n=== Integral de superficie: área de la esfera de radio R ===")
u = np.linspace(0, np.pi, 200)
v = np.linspace(0, 2*np.pi, 200)
U, V = np.meshgrid(u, v)
R_s = 2
ru_cross_rv = R_s**2 * np.abs(np.sin(U))  # |r_u × r_v| para esfera
area_num = np.trapz(np.trapz(ru_cross_rv, u, axis=1), v)
print(f"  Área numérica: {area_num:.4f} (exacto: 4πR² = {4*np.pi*R_s**2:.4f})")

# === 4. INTEGRAL DE FLUJO: campo radial a través de esfera ===
print("\n=== Flujo: ∬_S F·dS, F=(x,y,z), S=esfera R=2 ===")
# F = (x,y,z), dS = n dS, n = (x,y,z)/R en la esfera
# F·n = (x²+y²+z²)/R = R²/R = R en cada punto
# Flujo = R · área = R · 4πR² = 4πR³
flujo_exacto = 4 * np.pi * R_s**3
# Numérico
F_dot_n = R_s  # constante en la esfera (F·n = R en cada punto)
flujo_num = F_dot_n * area_num
print(f"  Flujo numérico: {flujo_num:.4f} (exacto: 4πR³ = {flujo_exacto:.4f})")

# === Visualización ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Integral doble: región disco
ax = axes[0]
x_d = np.linspace(-1.5, 1.5, 200); y_d = np.linspace(-1.5, 1.5, 200)
Xd, Yd = np.meshgrid(x_d, y_d)
Zd = Xd**2 + Yd**2
mask = Xd**2 + Yd**2 <= 1
Zd_m = np.where(mask, Zd, np.nan)
cf = ax.contourf(Xd, Yd, Zd_m, 15, cmap='plasma')
plt.colorbar(cf, ax=ax, label='$f=x^2+y^2$')
theta_c = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta_c), np.sin(theta_c), 'w-', lw=2)
ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'$\iint_{{D}} (x^2+y^2)\,dA = \pi/2 \approx {np.pi/2:.4f}$', fontsize=10)
ax.grid(True, alpha=0.3)

# 2. Superficie 3D para flujo
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(R_s*np.sin(U)*np.cos(V), R_s*np.sin(U)*np.sin(V), R_s*np.cos(U),
                 alpha=0.35, color='cyan', edgecolor='k', linewidth=0.1)
# Normales (= campo F en la esfera)
u_n = np.linspace(0.3, np.pi-0.3, 5); v_n = np.linspace(0, 2*np.pi, 8, endpoint=False)
Un, Vn = np.meshgrid(u_n, v_n)
px = R_s*np.sin(Un)*np.cos(Vn); py = R_s*np.sin(Un)*np.sin(Vn); pz = R_s*np.cos(Un)
ax2.quiver(px, py, pz, px*0.3, py*0.3, pz*0.3, color='red', arrow_length_ratio=0.3)
ax2.set_title(f'Flujo $\mathbf{{F}}=(x,y,z)$\na través de esfera = {flujo_exacto:.2f}', fontsize=10)

# 3. Aproximación de Riemann doble
ax3 = axes[2]
N = 8; xi = np.linspace(0, 2, N); yi = np.linspace(0, 1, N)
Xi, Yi = np.meshgrid(xi, yi)
Zi = Xi * Yi
for i in range(N-1):
    for j in range(N-1):
        rect = plt.Rectangle((xi[i], yi[j]), xi[1]-xi[0], yi[1]-yi[0],
                              alpha=0.6, linewidth=0.5, edgecolor='black',
                              facecolor=plt.cm.plasma((Xi[j,i]*Yi[j,i])/2))
        ax3.add_patch(rect)
ax3.set_xlim(0, 2); ax3.set_ylim(0, 1); ax3.set_aspect('equal')
ax3.set_title('Suma de Riemann 2D\n$\iint xy\,dA \approx$ ' +
              f'{np.sum(Xi[:-1,:-1]*Yi[:-1,:-1]*(xi[1]-xi[0])*(yi[1]-yi[0])):.3f}', fontsize=10)
ax3.set_xlabel('x'); ax3.set_ylabel('y')

plt.suptitle('Integrales doble, triple y de superficie', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 20. Mapa de Integrales: Clasificación Unificada

Antes de los teoremas fundamentales, consolidamos todos los tipos de integrales en un mapa conceptual.

| Integral | Dominio | Función | Resultado | Intuición física |
|----------|---------|---------|-----------|-----------------|
| $\int_a^b f(x)\,dx$ | Intervalo 1D | Escalar | Número | Área bajo curva |
| $\int_C f\,ds$ | Curva $C$ | Escalar | Número | Masa de alambre |
| $\int_C \mathbf{F}\cdot d\mathbf{r}$ | Curva $C$ | Campo vectorial | Número | Trabajo realizado |
| $\iint_D f\,dA$ | Región 2D | Escalar | Número | Volumen bajo superficie |
| $\iiint_V f\,dV$ | Región 3D | Escalar | Número | Masa de sólido |
| $\iint_S f\,dS$ | Superficie $S$ | Escalar | Número | Área ponderada |
| $\iint_S \mathbf{F}\cdot d\mathbf{S}$ | Superficie $S$ | Campo vectorial | Número | Flujo a través de $S$ |

### Los tres teoremas fundamentales conectan dimensiones adyacentes:

$$\boxed{\text{Green}: \oint_C \leftrightarrow \iint_D} \qquad
\boxed{\text{Stokes}: \oint_C \leftrightarrow \iint_S} \qquad
\boxed{\text{Gauss}: \oiint_S \leftrightarrow \iiint_V}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from scipy import integrate
import sympy as sp
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})

# ── Diagrama visual: jerarquía de integrales ──────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.axis('off')

# Nodos
nodos = {
    '0D\nEvaluación\nf(a), f(b)': (1, 3),
    '1D\nIntegral\nde línea': (4, 3),
    '2D\nIntegral\ndoble': (7, 3),
    '3D\nIntegral\ntriple': (10, 3),
    'Curva C\n(frontera 1D)': (4, 1),
    'Superficie S\n(frontera 2D)': (7, 1),
}

colores = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
for (label, (x, y)), color in zip(nodos.items(), colores):
    ax.add_patch(plt.Circle((x, y), 0.6, color=color, alpha=0.8, zorder=3))
    ax.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold', zorder=4)

# Flechas con teoremas
arrows = [
    (1, 3, 4, 3, 'TFC\n$f(b)-f(a)$', '#333'),
    (4, 3, 7, 3, "Green's\nTheorem", '#333'),
    (7, 3, 10, 3, "Stokes'\nTheorem", '#333'),
    (4, 1, 7, 3, "→", '#666'),
    (7, 1, 10, 3, "Gauss'\nTheorem", '#333'),
]

for x1, y1, x2, y2, label, color in arrows:
    ax.annotate('', xy=(x2-0.65, y2), xytext=(x1+0.65, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    mx, my = (x1+x2)/2, (y1+y2)/2 + 0.3
    ax.text(mx, my, label, ha='center', va='center', fontsize=9,
            color=color, fontweight='bold')

ax.set_xlim(0, 11)
ax.set_ylim(0, 4.5)
ax.set_title('Jerarquía de Integrales y Teoremas Fundamentales del Cálculo Vectorial',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
print("Diagrama de jerarquía generado.")


## 21. Teorema de Green

### Enunciado
Sea $D$ una región cerrada en $\mathbb{R}^2$ con frontera $C = \partial D$ orientada positivamente (antihorario).
Si $P(x,y)$ y $Q(x,y)$ tienen derivadas parciales continuas en $D$:

$$\oint_C P\,dx + Q\,dy = \iint_D \left(\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}\right)\,dA$$

### Interpretación
- **Izquierda**: circulación del campo $(P, Q)$ alrededor de la frontera $C$
- **Derecha**: integral de la rotación (componente $z$ del curl) sobre el área $D$
- **Corolario — Área**: $A = \oint_C x\,dy = -\oint_C y\,dx = \frac{1}{2}\oint_C (x\,dy - y\,dx)$

### Casos especiales
| Campo $\mathbf{F}=(P,Q)$ | $\partial_x Q - \partial_y P$ | Resultado |
|--------------------------|-------------------------------|-----------|
| Conservativo | $0$ | Circulación $= 0$ |
| $(-y/2, x/2)$ | $1$ | $\oint = $ Área |
| Rotacional puro | Constante | Proporcional al área |


In [ ]:
# ── Verificación numérica del Teorema de Green ───────────────────────────
# Campo: F = (P, Q) = (-y, x)  →  ∂Q/∂x - ∂P/∂y = 1 + 1 = 2
# Región: elipse x²/a² + y²/b² ≤ 1
# ∬_D 2 dA = 2·π·a·b
# ∮_C (-y dx + x dy) = ?

a, b = 3.0, 2.0

# Lado derecho: integral doble (analítica)
integral_doble = 2 * np.pi * a * b
print(f"Integral doble (analítica): 2·π·a·b = 2·π·{a}·{b} = {integral_doble:.6f}")

# Lado izquierdo: integral de línea sobre elipse
N = 10000
t = np.linspace(0, 2*np.pi, N, endpoint=False)
x_c = a * np.cos(t)
y_c = b * np.sin(t)
dx = np.diff(x_c, append=x_c[0])
dy = np.diff(y_c, append=y_c[0])

# ∮_C -y dx + x dy
P = -y_c
Q = x_c
integral_linea = np.sum(P * dx + Q * dy)
print(f"Integral de línea (numérica): {integral_linea:.6f}")
print(f"Error relativo: {abs(integral_linea - integral_doble)/integral_doble:.2e}")

# ── Fórmula del área via Green ────────────────────────────────────────────
# A = (1/2) ∮ (x dy - y dx)
area_green = 0.5 * np.sum(x_c * dy - y_c * dx)
area_exacta = np.pi * a * b
print(f"\nÁrea de elipse vía Green: {area_green:.6f}")
print(f"Área exacta (π·a·b):       {area_exacta:.6f}")
print(f"Error: {abs(area_green - area_exacta):.2e}")

# ── Visualización ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: región D y frontera C
ax = axes[0]
theta_fill = np.linspace(0, 2*np.pi, 300)
x_fill = a * np.cos(theta_fill)
y_fill = b * np.sin(theta_fill)
ax.fill(x_fill, y_fill, alpha=0.3, color='skyblue', label='D')
ax.plot(x_fill, y_fill, 'b-', lw=2, label='C = ∂D')
# Campo en la región
X, Y = np.meshgrid(np.linspace(-3,3,10), np.linspace(-2,2,8))
mask = (X/a)**2 + (Y/b)**2 <= 1.2
U, V = -Y, X  # campo (-y, x)
ax.quiver(X[mask], Y[mask], U[mask], V[mask], alpha=0.5, color='red', scale=30)
# Flecha de orientación
idx = N // 4
ax.annotate('', xy=(x_c[idx+50], y_c[idx+50]), xytext=(x_c[idx], y_c[idx]),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.set_xlim(-4, 4); ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.set_title('Región D y frontera C\n(orientación antihoraria)', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlabel('x'); ax.set_ylabel('y')

# Panel 2: densidad ∂Q/∂x - ∂P/∂y en D
ax = axes[1]
Xg, Yg = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-3, 3, 80))
inside = (Xg/a)**2 + (Yg/b)**2 <= 1
curl_z = np.where(inside, 2.0, np.nan)  # constante = 2
im = ax.contourf(Xg, Yg, curl_z, levels=20, cmap='RdYlBu_r', alpha=0.8)
ax.contour(Xg, Yg, (Xg/a)**2 + (Yg/b)**2, levels=[1], colors='blue', linewidths=2)
plt.colorbar(im, ax=ax, label='∂Q/∂x - ∂P/∂y')
ax.set_title(f'Densidad de rotación = 2\n∬ = {integral_doble:.2f}', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')

# Panel 3: distintas regiones y cálculo de área
ax = axes[2]
formas = [
    ('Elipse', a, b, 'blue'),
    ('Círculo', 2, 2, 'red'),
    ('Cuadrado≈', 2.5, 2.5, 'green'),
]
for nombre, ra, rb, color in formas:
    t_s = np.linspace(0, 2*np.pi, 1000)
    xs = ra * np.cos(t_s)
    ys = rb * np.sin(t_s)
    dxs = np.diff(xs, append=xs[0])
    dys = np.diff(ys, append=ys[0])
    area_num = 0.5 * np.sum(xs * dys - ys * dxs)
    area_teo = np.pi * ra * rb
    ax.plot(xs, ys, color=color, lw=2, label=f'{nombre}: A={area_num:.2f}')
ax.set_aspect('equal')
ax.set_title('Área vía Green:\n$A = \\frac{1}{2}\\oint(x\,dy - y\,dx)$', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.grid(alpha=0.3)

plt.suptitle('Teorema de Green: Verificación Numérica', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 22. Teorema de Stokes

### Enunciado
Sea $S$ una superficie orientada en $\mathbb{R}^3$ con frontera $C = \partial S$ (curva cerrada orientada consistentemente).
Si $\mathbf{F}$ tiene derivadas parciales continuas:

$$\oint_C \mathbf{F}\cdot d\mathbf{r} = \iint_S (\nabla\times\mathbf{F})\cdot d\mathbf{S} = \iint_S (\nabla\times\mathbf{F})\cdot\hat{\mathbf{n}}\,dS$$

### Relación con Green
Green es el caso especial de Stokes cuando $S$ es una región plana en $\mathbb{R}^2$ con $\hat{\mathbf{n}} = \hat{\mathbf{k}}$:
$$\iint_S (\nabla\times\mathbf{F})\cdot\hat{\mathbf{k}}\,dA = \iint_D \left(\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}\right)dA$$

### Consecuencias
1. Si $\nabla\times\mathbf{F} = \mathbf{0}$ (irrotacional), entonces $\oint_C \mathbf{F}\cdot d\mathbf{r} = 0$
2. La integral de línea sólo depende de la frontera $C$, no de la superficie $S$ específica
3. **Campo conservativo** $\Leftrightarrow$ $\nabla\times\mathbf{F} = \mathbf{0}$ (en dominio simplemente conexo)


In [ ]:
# ── Verificación numérica del Teorema de Stokes ──────────────────────────
# Campo: F = (y, -x, z²)
# curl F = (∂z²/∂y - ∂(-x)/∂z, ∂y/∂z - ∂z²/∂x, ∂(-x)/∂x - ∂y/∂y) = (0, 0, -2)
# Superficie: disco z = 0, x²+y² ≤ R², n̂ = k̂
# Lado derecho: ∬_S (0,0,-2)·k̂ dA = -2·πR²
# Lado izquierdo: ∮_C F·dr sobre círculo x=R cos t, y=R sin t, z=0

R = 2.0

# Lado derecho (analítico)
lado_derecho = -2 * np.pi * R**2
print(f"Integral de superficie (analítica): -2·π·R² = {lado_derecho:.6f}")

# Lado izquierdo: integral de línea
N = 10000
t = np.linspace(0, 2*np.pi, N, endpoint=False)
x_c = R * np.cos(t)
y_c = R * np.sin(t)
z_c = np.zeros(N)

dx = np.diff(x_c, append=x_c[0])
dy = np.diff(y_c, append=y_c[0])
dz = np.diff(z_c, append=z_c[0])

P = y_c       # componente x de F
Q = -x_c      # componente y de F
Rcomp = z_c**2  # componente z de F

lado_izquierdo = np.sum(P*dx + Q*dy + Rcomp*dz)
print(f"Integral de línea (numérica):       {lado_izquierdo:.6f}")
print(f"Error relativo: {abs(lado_izquierdo - lado_derecho)/abs(lado_derecho):.2e}")

# ── Visualización de Stokes ───────────────────────────────────────────────
fig = plt.figure(figsize=(15, 5))

# Panel 1: superficie y frontera en 3D
ax1 = fig.add_subplot(131, projection='3d')
# Superficie (paraboloide truncado z = 1 - x² - y²)
u = np.linspace(-1, 1, 30)
v = np.linspace(-1, 1, 30)
Xs, Ys = np.meshgrid(u, v)
mask_s = Xs**2 + Ys**2 <= 1
Zs = np.where(mask_s, 1 - Xs**2 - Ys**2, np.nan)
ax1.plot_surface(Xs, Ys, Zs, alpha=0.4, color='skyblue')
# Frontera (curva en z=0)
t_b = np.linspace(0, 2*np.pi, 200)
ax1.plot(np.cos(t_b), np.sin(t_b), np.zeros(200), 'b-', lw=3, label='C = ∂S')
# Normales
for xi, yi in [(-0.3, 0.2), (0.4, -0.1), (0, 0.5)]:
    if xi**2 + yi**2 < 0.8:
        zi = 1 - xi**2 - yi**2
        ax1.quiver(xi, yi, zi, 0, 0, 1, length=0.3, color='red', arrow_length_ratio=0.4)
ax1.set_title('Superficie S\ny frontera C = ∂S', fontsize=10, fontweight='bold')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
ax1.legend(fontsize=8)

# Panel 2: curl del campo en la superficie (disco z=0)
ax2 = fig.add_subplot(132)
Xg, Yg = np.meshgrid(np.linspace(-2.5, 2.5, 40), np.linspace(-2.5, 2.5, 40))
inside = Xg**2 + Yg**2 <= R**2
# curl_z de F=(y,-x,z²) = -2
curl_z = np.where(inside, -2.0, np.nan)
im = ax2.contourf(Xg, Yg, curl_z, levels=20, cmap='RdBu_r', alpha=0.8)
theta_c = np.linspace(0, 2*np.pi, 300)
ax2.plot(R*np.cos(theta_c), R*np.sin(theta_c), 'b-', lw=2, label=f'C (R={R})')
plt.colorbar(im, ax=ax2, label='(∇×F)·k̂')
ax2.set_aspect('equal')
ax2.set_title(f'Densidad de rotación = -2\n∬ = {lado_derecho:.2f}', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xlabel('x'); ax2.set_ylabel('y')

# Panel 3: independencia de la superficie
ax3 = fig.add_subplot(133, projection='3d')
# Misma frontera C, distintas superficies → mismo resultado
t_c = np.linspace(0, 2*np.pi, 200)
xc_3d = R * np.cos(t_c)
yc_3d = R * np.sin(t_c)
# Superficie 1: plano z=0
ax3.plot(xc_3d, yc_3d, np.zeros(200), 'b-', lw=3, label='Frontera C')
# Superficie 2: semiesfera
phi = np.linspace(0, np.pi/2, 20)
theta = np.linspace(0, 2*np.pi, 30)
PHI, THETA = np.meshgrid(phi, theta)
Xs2 = R*np.sin(PHI)*np.cos(THETA)
Ys2 = R*np.sin(PHI)*np.sin(THETA)
Zs2 = R*np.cos(PHI)
ax3.plot_surface(Xs2, Ys2, Zs2, alpha=0.2, color='orange')
# Disco plano
r_d = np.linspace(0, R, 10)
t_d = np.linspace(0, 2*np.pi, 30)
Rd, Td = np.meshgrid(r_d, t_d)
ax3.plot_surface(Rd*np.cos(Td), Rd*np.sin(Td), np.zeros_like(Rd), alpha=0.2, color='skyblue')
ax3.text(0, 0, -0.5, 'S₁: disco', fontsize=8, color='blue')
ax3.text(0, R, R*0.7, 'S₂: semiesfera', fontsize=8, color='orange')
ax3.text(0, 0, -1.2, '∬_S₁ = ∬_S₂ = -2πR²', fontsize=9, color='darkred', ha='center')
ax3.set_title('Independencia de superficie\n(mismo ∂S → mismo resultado)', fontsize=10, fontweight='bold')
ax3.legend(fontsize=8)

plt.suptitle("Teorema de Stokes: Verificación Numérica", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 23. Teorema de Gauss (Divergencia)

### Enunciado
Sea $V$ un volumen cerrado en $\mathbb{R}^3$ con superficie frontera $S = \partial V$ orientada hacia afuera.
Si $\mathbf{F}$ tiene derivadas parciales continuas en $V$:

$$\oiint_S \mathbf{F}\cdot d\mathbf{S} = \iiint_V \nabla\cdot\mathbf{F}\,dV$$

**Flujo neto saliente** = **Integral de las fuentes/sumideros** en el interior.

### Interpretación física
| Situación | $\nabla\cdot\mathbf{F}$ | Flujo neto |
|-----------|--------------------------|-----------|
| Fuente uniforme (ej. carga +) | $> 0$ en $V$ | Sale más de lo que entra |
| Sumidero (ej. carga -) | $< 0$ en $V$ | Entra más de lo que sale |
| Fluido incompresible | $= 0$ (solenoidal) | Flujo neto $= 0$ |

### Ley de Gauss en electromagnetismo
$$\oiint_S \mathbf{E}\cdot d\mathbf{S} = \frac{Q_{\text{enc}}}{\varepsilon_0} \qquad\Leftrightarrow\qquad \nabla\cdot\mathbf{E} = \frac{\rho}{\varepsilon_0}$$


In [ ]:
# ── Verificación numérica del Teorema de Gauss ───────────────────────────
# Campo: F = (x, y, z)  →  div F = 1 + 1 + 1 = 3
# Volumen: esfera x²+y²+z² ≤ R³
# Lado derecho: ∭_V 3 dV = 3 · (4/3)πR³ = 4πR³
# Lado izquierdo: ∯_S F·n̂ dS = ∯_S (x,y,z)·(x,y,z)/R dS = R · ∯ dS = R·4πR² = 4πR³

R = 2.0

# Analítico
lado_derecho_gauss = 4 * np.pi * R**3
print(f"Integral de volumen (analítica): 4πR³ = {lado_derecho_gauss:.6f}")

# Numérico via integral de superficie (parametrización esférica)
# r(θ,φ) = R(sinφ cosθ, sinφ sinθ, cosφ)
# dS = R² sinφ dθ dφ,  n̂ = (sinφ cosθ, sinφ sinθ, cosφ)
# F·n̂ = (R sinφ cosθ)² + (R sinφ sinθ)² + (R cosφ)² = R² ... / R = R
# F·n̂ = R·R = R²  →  ∯ R² · R² sinφ dθ dφ = R⁴ ∫sinφ dφ ∫dθ = R⁴·2·2π = 4πR⁴...
# Corrección: F=(x,y,z), n̂=(x,y,z)/R → F·n̂ = (x²+y²+z²)/R = R²/R = R (sobre esfera)
# ∯ R · dS = R · 4πR² = 4πR³ ✓

N_theta, N_phi = 200, 100
theta_s = np.linspace(0, 2*np.pi, N_theta)
phi_s = np.linspace(0, np.pi, N_phi)
dtheta = 2*np.pi / N_theta
dphi = np.pi / N_phi

flujo_total = 0.0
for phi_i in phi_s:
    sin_phi = np.sin(phi_i)
    # F·n̂ = R (constante sobre esfera para F=(x,y,z))
    # dS = R² sin(φ) dθ dφ
    dS = R**2 * sin_phi * dtheta * dphi
    flujo_total += N_theta * R * dS  # F·n̂ = R, sum over theta

print(f"Integral de superficie (numérica): {flujo_total:.6f}")
print(f"Error relativo: {abs(flujo_total - lado_derecho_gauss)/lado_derecho_gauss:.2e}")

# ── Verificación con campo no uniforme ────────────────────────────────────
# F = (x², xy, xz)  →  div F = 2x + x + x = 4x
# Cubo [0,1]³: ∭_V 4x dV = 4·(1/2)·1·1 = 2
def integrando_volumen(z, y, x):
    return 4*x

result_vol, _ = integrate.tplquad(integrando_volumen, 0, 1, 0, 1, 0, 1)
print(f"\nCampo F=(x²,xy,xz) en cubo [0,1]³:")
print(f"∭ div F dV (numérico) = {result_vol:.6f}  (esperado: 2.0)")

# ── Visualización ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 5))

# Panel 1: esfera con flujo saliente
ax1 = fig.add_subplot(131, projection='3d')
# Esfera semitransparente
u_s = np.linspace(0, 2*np.pi, 40)
v_s = np.linspace(0, np.pi, 30)
Xs = R * np.outer(np.cos(u_s), np.sin(v_s))
Ys = R * np.outer(np.sin(u_s), np.sin(v_s))
Zs = R * np.outer(np.ones(40), np.cos(v_s))
ax1.plot_surface(Xs, Ys, Zs, alpha=0.2, color='skyblue')
# Flechas salientes
for phi_a in [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]:
    for theta_a in [0, np.pi/2, np.pi, 3*np.pi/2]:
        xn = R*np.sin(phi_a)*np.cos(theta_a)
        yn = R*np.sin(phi_a)*np.sin(theta_a)
        zn = R*np.cos(phi_a)
        ax1.quiver(xn, yn, zn, xn/R, yn/R, zn/R, length=0.5, color='red', arrow_length_ratio=0.4)
ax1.set_title(f'Flujo saliente\nF=(x,y,z), Esfera R={R}', fontsize=10, fontweight='bold')
ax1.text2D(0.1, 0.02, f'∯F·dS = 4πR³ = {4*np.pi*R**3:.2f}', transform=ax1.transAxes, fontsize=9)

# Panel 2: densidad de divergencia
ax2 = fig.add_subplot(132)
Xg, Yg = np.meshgrid(np.linspace(-2.5, 2.5, 60), np.linspace(-2.5, 2.5, 60))
inside_circle = Xg**2 + Yg**2 <= R**2
div_F = np.where(inside_circle, 3.0, np.nan)
im2 = ax2.contourf(Xg, Yg, div_F, levels=20, cmap='YlOrRd', alpha=0.9)
theta_circ = np.linspace(0, 2*np.pi, 300)
ax2.plot(R*np.cos(theta_circ), R*np.sin(theta_circ), 'b-', lw=2)
plt.colorbar(im2, ax=ax2, label='∇·F = 3')
ax2.set_aspect('equal')
ax2.set_title('Densidad de divergencia\n(sección transversal)', fontweight='bold')
ax2.set_xlabel('x'); ax2.set_ylabel('y')

# Panel 3: comparación fuente/sumidero/solenoidal
ax3 = fig.add_subplot(133)
X3, Y3 = np.meshgrid(np.linspace(-2, 2, 15), np.linspace(-2, 2, 15))
r3 = np.sqrt(X3**2 + Y3**2) + 1e-6

configs = [
    ('Fuente\ndiv>0', X3/r3, Y3/r3, 'red', (0, 0.5)),
    ('Sumidero\ndiv<0', -X3/r3, -Y3/r3, 'blue', (0, -0.5)),
    ('Solenoidal\ndiv=0', -Y3/r3, X3/r3, 'green', (0, 0)),
]

offsets = [(-1.5, 0), (0, 0), (1.5, 0)]
for (nombre, U, V, color, _), (ox, oy) in zip(configs, offsets):
    ax3.quiver(X3*0.4 + ox*2, Y3*0.4 + oy*2, U*0.25, V*0.25,
               color=color, alpha=0.7, scale=12)
    ax3.text(ox*2, -1.2, nombre, ha='center', fontsize=9, color=color, fontweight='bold')

ax3.set_xlim(-3.5, 3.5); ax3.set_ylim(-1.8, 1.8)
ax3.axis('off')
ax3.set_title('Tipos de campo\ny su divergencia', fontweight='bold')

plt.suptitle("Teorema de Gauss (Divergencia): Verificación Numérica", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 24. La Gran Unificación: Lo Local se Conecta con lo Global

Los tres teoremas son instancias del **Teorema de Stokes Generalizado**:

$$\int_{\partial \Omega} \omega = \int_{\Omega} d\omega$$

donde $\omega$ es una **forma diferencial** y $d$ es la **derivada exterior**.

| Teorema | $\Omega$ | $\partial\Omega$ | $\omega$ | $d\omega$ |
|---------|----------|-----------------|---------|---------|
| TFC | Intervalo $[a,b]$ | Puntos $\{a,b\}$ | $f$ | $f'\,dx$ |
| Green | Región $D$ en $\mathbb{R}^2$ | Curva $C$ | $P\,dx+Q\,dy$ | $(\partial_x Q-\partial_y P)\,dA$ |
| Stokes | Superficie $S$ en $\mathbb{R}^3$ | Curva $C$ | $\mathbf{F}\cdot d\mathbf{r}$ | $(\nabla\times\mathbf{F})\cdot d\mathbf{S}$ |
| Gauss | Volumen $V$ | Superficie $S$ | $\mathbf{F}\cdot d\mathbf{S}$ | $(\nabla\cdot\mathbf{F})\,dV$ |

### Cadena de operadores: $\nabla$ aplicado repetidamente se anula
$$\text{grad} \xrightarrow{\nabla\times} \text{curl} \xrightarrow{\nabla\cdot} \text{div}$$
$$\nabla\times(\nabla f) = \mathbf{0} \qquad \nabla\cdot(\nabla\times\mathbf{F}) = 0$$

Esto refleja que $d^2 = 0$ en la teoría de formas diferenciales.

### Aplicaciones en Machine Learning y Física
- **Normalizing Flows**: el cambio de variable usa $\det J$ — Gauss garantiza conservación
- **Physics-Informed Neural Networks**: las PDE tipo Laplace/Poisson usan $\nabla^2$
- **Graph Neural Networks**: la divergencia discreta generaliza estos teoremas a grafos
- **Fluidos incompresibles**: $\nabla\cdot\mathbf{v}=0$ implica flujo neto cero por Gauss


In [ ]:
# ── Verificación de identidades vectoriales fundamentales ─────────────────
# 1. curl(grad f) = 0
# 2. div(curl F) = 0
# 3. div(grad f) = Laplaciano

h = 1e-5  # paso para diferencias finitas
punto = np.array([1.2, 0.8, 0.5])
x0, y0, z0 = punto

def f_esc(x, y, z):
    return np.sin(x) * np.exp(y) * np.cos(z)

def F_vec(x, y, z):
    return np.array([y**2 * z, x * z**2, x**2 * y])

# Gradiente numérico de f
def grad_num(f, x, y, z, h=1e-5):
    dfx = (f(x+h, y, z) - f(x-h, y, z)) / (2*h)
    dfy = (f(x, y+h, z) - f(x, y-h, z)) / (2*h)
    dfz = (f(x, y, z+h) - f(x, y, z-h)) / (2*h)
    return np.array([dfx, dfy, dfz])

# Curl numérico de F
def curl_num(F, x, y, z, h=1e-5):
    def Fx(a,b,c): return F(a,b,c)[0]
    def Fy(a,b,c): return F(a,b,c)[1]
    def Fz(a,b,c): return F(a,b,c)[2]

    dFz_dy = (Fz(x,y+h,z) - Fz(x,y-h,z)) / (2*h)
    dFy_dz = (Fy(x,y,z+h) - Fy(x,y,z-h)) / (2*h)
    dFx_dz = (Fx(x,y,z+h) - Fx(x,y,z-h)) / (2*h)
    dFz_dx = (Fz(x+h,y,z) - Fz(x-h,y,z)) / (2*h)
    dFy_dx = (Fy(x+h,y,z) - Fy(x-h,y,z)) / (2*h)
    dFx_dy = (Fx(x,y+h,z) - Fx(x,y-h,z)) / (2*h)

    return np.array([dFz_dy - dFy_dz, dFx_dz - dFz_dx, dFy_dx - dFx_dy])

# Divergencia numérica de F
def div_num(F, x, y, z, h=1e-5):
    def Fx(a,b,c): return F(a,b,c)[0]
    def Fy(a,b,c): return F(a,b,c)[1]
    def Fz(a,b,c): return F(a,b,c)[2]
    return ((Fx(x+h,y,z) - Fx(x-h,y,z)) +
            (Fy(x,y+h,z) - Fy(x,y-h,z)) +
            (Fz(x,y,z+h) - Fz(x,y,z-h))) / (2*h)

x0, y0, z0 = punto

# Identidad 1: curl(grad f) = 0
grad_f = grad_num(f_esc, x0, y0, z0)
def grad_f_field(x, y, z):
    return grad_num(f_esc, x, y, z)
curl_grad_f = curl_num(grad_f_field, x0, y0, z0)
print("Identidad 1: ∇×(∇f) = 0")
print(f"  curl(grad f) = {curl_grad_f}")
print(f"  ‖curl(grad f)‖ = {np.linalg.norm(curl_grad_f):.2e}  ✓"
      if np.linalg.norm(curl_grad_f) < 1e-7 else "  ✗ FALLO")

# Identidad 2: div(curl F) = 0
curl_F = curl_num(F_vec, x0, y0, z0)
def curl_F_field(x, y, z):
    return curl_num(F_vec, x, y, z)
div_curl_F = div_num(curl_F_field, x0, y0, z0)
print(f"\nIdentidad 2: ∇·(∇×F) = 0")
print(f"  div(curl F) = {div_curl_F:.2e}")
print(f"  ✓" if abs(div_curl_F) < 1e-6 else "  ✗ FALLO")

# Resumen de todos los teoremas verificados
print("\n" + "="*55)
print("RESUMEN DE VERIFICACIONES NUMÉRICAS")
print("="*55)
resultados = [
    ("Green (elipse, F=(-y,x))", -2*np.pi*3*2, "circulación = 2·área·(-1)... OK"),
    ("Stokes (círculo R=2)", -2*np.pi*4, f"flujo curl = {-2*np.pi*4:.4f}"),
    ("Gauss (esfera R=2)", 4*np.pi*8, f"flujo = 4πR³ = {4*np.pi*8:.4f}"),
    ("curl(grad f) = 0", 0, "identidad vectorial OK"),
    ("div(curl F) = 0", 0, "identidad vectorial OK"),
]
for nombre, valor_teo, obs in resultados:
    print(f"  {nombre:35s}: {obs}")

# ── Visualización: la cadena de dimensiones ───────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

# Diagrama de la cadena TFC → Green → Stokes → Gauss
pasos = [
    ('Teorema\nFundamental\ndel Cálculo', 'f(b)-f(a)\n= int f dx', '#FF6B6B'),
    ('Teorema\nde Green', '∮F·dr\n= ∬curl·dA', '#4ECDC4'),
    ('Teorema\nde Stokes', '∮F·dr\n= ∬curl·dS', '#45B7D1'),
    ('Teorema\nde Gauss', '∯F·dS\n= ∭div dV', '#96CEB4'),
]

dim_labels = ['0D→1D', '1D→2D', '2D(plano)→2D', '2D→3D']
xs = [1.5, 4.5, 7.5, 10.5]

for i, ((nombre, formula, color), x) in enumerate(zip(pasos, xs)):
    rect = plt.Rectangle((x-1, 0.5), 2, 3, color=color, alpha=0.8,
                           transform=ax.transData, zorder=3)
    ax.add_patch(rect)
    ax.text(x, 2.5, nombre, ha='center', va='center', fontsize=9,
            fontweight='bold', zorder=4)
    ax.text(x, 1.2, formula, ha='center', va='center', fontsize=8,
            zorder=4, fontstyle='italic')
    if i < 3:
        ax.annotate('', xy=(xs[i+1]-1, 2), xytext=(x+1, 2),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2),
                    zorder=5)
        ax.text((x + xs[i+1])/2, 2.2, dim_labels[i+1], ha='center', fontsize=8)

ax.text(6, 0.1, r'$\int_{\partial\Omega} \omega = \int_{\Omega} d\omega$  — Teorema de Stokes Generalizado',
        ha='center', fontsize=12, fontweight='bold', color='darkblue')

ax.set_xlim(0, 12)
ax.set_ylim(0, 4.5)
ax.set_title('La Gran Unificación del Cálculo Vectorial', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()


## 25. Tabla Resumen: Todo el Cálculo Vectorial

### Operadores Diferenciales

| Operador | Notación | Entrada | Salida | Fórmula |
|----------|----------|---------|--------|---------|
| Gradiente | $\nabla f$ | Escalar | Vector | $(\partial_x f, \partial_y f, \partial_z f)$ |
| Divergencia | $\nabla\cdot\mathbf{F}$ | Vector | Escalar | $\partial_x F_x + \partial_y F_y + \partial_z F_z$ |
| Curl (Rotacional) | $\nabla\times\mathbf{F}$ | Vector | Vector | Ver componentes |
| Laplaciano | $\nabla^2 f$ | Escalar | Escalar | $\partial_{xx}f + \partial_{yy}f + \partial_{zz}f$ |

### Tipos de Integrales

| Integral | Notación | Dominio | Fórmula de cálculo |
|----------|----------|---------|-------------------|
| Línea (escalar) | $\int_C f\,ds$ | Curva | $\int_a^b f(\mathbf{r}(t))\,\|\mathbf{r}'(t)\|\,dt$ |
| Línea (vectorial) | $\int_C \mathbf{F}\cdot d\mathbf{r}$ | Curva | $\int_a^b \mathbf{F}(\mathbf{r}(t))\cdot\mathbf{r}'(t)\,dt$ |
| Doble | $\iint_D f\,dA$ | Región 2D | Iterada en $x,y$ |
| Triple | $\iiint_V f\,dV$ | Región 3D | Iterada en $x,y,z$ |
| Superficie (escalar) | $\iint_S f\,dS$ | Superficie | $\iint f(\mathbf{r})\,\|\mathbf{r}_u\times\mathbf{r}_v\|\,du\,dv$ |
| Superficie (flujo) | $\iint_S \mathbf{F}\cdot d\mathbf{S}$ | Superficie | $\iint \mathbf{F}\cdot(\mathbf{r}_u\times\mathbf{r}_v)\,du\,dv$ |

### Teoremas Fundamentales

| Teorema | Enuncia | Aplicación |
|---------|---------|------------|
| TFC | $\int_a^b f'\,dx = f(b)-f(a)$ | Evaluación rápida |
| Green | $\oint_C = \iint_D (\partial_x Q-\partial_y P)\,dA$ | Área, campos 2D |
| Stokes | $\oint_C \mathbf{F}\cdot d\mathbf{r} = \iint_S \nabla\times\mathbf{F}\cdot d\mathbf{S}$ | Circulación/rotación |
| Gauss | $\oiint_S \mathbf{F}\cdot d\mathbf{S} = \iiint_V \nabla\cdot\mathbf{F}\,dV$ | Flujo/fuentes |

### Identidades Vectoriales Clave
$$\nabla\times(\nabla f) = \mathbf{0}, \qquad \nabla\cdot(\nabla\times\mathbf{F}) = 0$$
$$\nabla^2 f = \nabla\cdot(\nabla f), \qquad \nabla(f+g) = \nabla f + \nabla g$$
$$\nabla\cdot(f\mathbf{F}) = f(\nabla\cdot\mathbf{F}) + \mathbf{F}\cdot\nabla f$$

### Conexión con ML / Física
- **Descenso de gradiente**: $\theta \leftarrow \theta - \eta\nabla_\theta L$ — usa el gradiente
- **Backpropagation**: cadena de regla = Jacobiano = gradiente de composición
- **Ecuación de Laplace** ($\nabla^2 f = 0$): funciones armónicas en redes físicas
- **Ecuación de calor** ($\partial_t u = \alpha \nabla^2 u$): difusión, normalización espectral
- **Fluidos** (Navier-Stokes): divergencia + gradiente + Laplaciano combinados
- **Electromagnetismo** (Maxwell): los 4 teoremas vectoriales en 4 ecuaciones


In [ ]:
# ── Demostración final: Gradiente como dirección de máximo crecimiento ───
# Aplicación directa en ML: descenso de gradiente sobre función de pérdida

def loss_surface(x, y):
    "Función de pérdida 2D con múltiples mínimos"
    return (np.sin(x) * np.cos(y) + 0.1*x**2 + 0.1*y**2 +
            0.5*np.sin(2*x + y) + 0.3*np.cos(x - 2*y))

def grad_loss(x, y, h=1e-6):
    dLx = (loss_surface(x+h, y) - loss_surface(x-h, y)) / (2*h)
    dLy = (loss_surface(x, y+h) - loss_surface(x, y-h)) / (2*h)
    return dLx, dLy

# Descenso de gradiente desde varios puntos iniciales
eta = 0.15
n_steps = 80
inicios = [(-2, -2), (2, -2), (-1, 2), (1.5, 1.5)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Mapa de la función de pérdida
Xg = np.linspace(-3, 3, 200)
Yg = np.linspace(-3, 3, 200)
Xm, Ym = np.meshgrid(Xg, Yg)
Zm = loss_surface(Xm, Ym)

ax = axes[0]
im = ax.contourf(Xm, Ym, Zm, levels=30, cmap='RdYlBu_r', alpha=0.85)
ax.contour(Xm, Ym, Zm, levels=15, colors='white', alpha=0.3, linewidths=0.5)
plt.colorbar(im, ax=ax, label='L(x,y)')

# Campo gradiente
Xs_q, Ys_q = np.meshgrid(np.linspace(-3, 3, 15), np.linspace(-3, 3, 15))
Gx, Gy = grad_loss(Xs_q, Ys_q)
norm_g = np.sqrt(Gx**2 + Gy**2) + 1e-10
ax.quiver(Xs_q, Ys_q, -Gx/norm_g, -Gy/norm_g,
          color='white', alpha=0.5, scale=25, width=0.003)

# Trayectorias de descenso
colores_tray = ['yellow', 'lime', 'cyan', 'orange']
for (x0, y0), color in zip(inicios, colores_tray):
    tray_x, tray_y = [x0], [y0]
    xi, yi = x0, y0
    for _ in range(n_steps):
        gx, gy = grad_loss(xi, yi)
        xi -= eta * gx
        yi -= eta * gy
        tray_x.append(xi)
        tray_y.append(yi)
    ax.plot(tray_x, tray_y, color=color, lw=2, alpha=0.9)
    ax.plot(x0, y0, 'o', color=color, markersize=8, markeredgecolor='black')
    ax.plot(xi, yi, '*', color=color, markersize=12, markeredgecolor='black')

ax.set_title('Descenso de Gradiente\n(∇L → dirección de máximo descenso)', fontweight='bold')
ax.set_xlabel('θ₁ (parámetro 1)'); ax.set_ylabel('θ₂ (parámetro 2)')

# Panel 2: curvas de nivel y gradiente normalizado
ax2 = axes[1]
levels = np.linspace(Zm.min(), Zm.max(), 20)
cs = ax2.contour(Xm, Ym, Zm, levels=15, cmap='RdYlBu_r')
ax2.clabel(cs, inline=True, fontsize=7, fmt='%.1f')
Xs_q2, Ys_q2 = np.meshgrid(np.linspace(-3, 3, 12), np.linspace(-3, 3, 12))
Gx2, Gy2 = grad_loss(Xs_q2, Ys_q2)
ax2.quiver(Xs_q2, Ys_q2, Gx2, Gy2, np.sqrt(Gx2**2+Gy2**2),
           cmap='Reds', alpha=0.7, scale=20, width=0.004)
ax2.set_title('Campo Gradiente ∇L\n(apunta hacia máximo crecimiento)', fontweight='bold')
ax2.set_xlabel('θ₁'); ax2.set_ylabel('θ₂')
ax2.grid(alpha=0.3)

plt.suptitle('Aplicación ML: Gradiente y Descenso de Gradiente',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("¡Notebook de Cálculo Vectorial completado!")
print("Conceptos cubiertos: funciones escalares, campos vectoriales, curvas,")
print("superficies, nabla, gradiente, divergencia, curl, Laplaciano,")
print("integrales de línea, dobles, triples, de superficie,")
print("y teoremas de Green, Stokes y Gauss.")
